# Auto-HKG x Unsloth — Local Inference on A100 (Colab Pro)

This notebook runs the Auto-HKG pipeline entirely on a local GPU using Unsloth
for optimized inference. No API key is required.

**Hardware target:** A100 40GB (Colab Pro / Pro+)

**What this notebook does:**
1. Verify that an A100 GPU is active
2. Install Unsloth and all required dependencies
3. Clone the Auto-HKG repository
4. Upload the dataset
5. Inspect available VRAM and select the appropriate model
6. Run the full pipeline
7. Monitor VRAM usage during inference
8. Visualize and download the output knowledge graph

---

### Model options for A100-40GB

| Alias | HuggingFace Model | Est. VRAM | Notes |
|---|---|---|---|
| `qwen3-14b` | Qwen3-14B-bnb-4bit | ~10 GB | Recommended. Fast and accurate JSON output. |
| `qwen3-32b` | Qwen3-32B-bnb-4bit | ~20 GB | Highest quality on A100-40GB. |
| `qwen2.5-32b` | Qwen2.5-32B-Instruct-bnb-4bit | ~20 GB | Strong instruction following. |
| `qwen3-14b-bf16` | Qwen3-14B (full bf16) | ~28 GB | Full precision. Best if VRAM allows. |
| `qwen3-32b-bf16` | Qwen3-32B (full bf16) | ~65 GB | A100-80GB only. |

All 4-bit models use NF4 quantization via bitsandbytes. Full bf16 models are loaded
without quantization and offer higher output fidelity at the cost of more VRAM.

In [ ]:
# ── Cell 0: Verify GPU ────────────────────────────────────────────────────────
# This cell checks that an A100 is active before proceeding.
# If a different GPU is detected, the notebook will still run but performance
# and model recommendations will differ from what is documented above.

import subprocess

result = subprocess.run(
    ['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv,noheader'],
    capture_output=True, text=True
)

if result.returncode != 0:
    raise RuntimeError(
        'GPU not detected. '
        'Enable GPU runtime via: Runtime > Change runtime type > A100 GPU'
    )

gpu_info = result.stdout.strip()
print(f'GPU detected: {gpu_info}')

if 'A100' not in gpu_info:
    print('Warning: This notebook is optimized for A100. '
          'Detected GPU may have different VRAM limits. Adjust model selection accordingly.')

In [ ]:
# ── Cell 1: Install Unsloth and dependencies ──────────────────────────────────
# Unsloth requires a specific installation command — do not use 'pip install unsloth'.
# The [colab-new] extra installs the version optimized for Colab environments.
# PyTorch is pinned to the cu124 index to match CUDA 12.4 on A100 instances.
# xformers is required for memory-efficient attention in Unsloth.

!pip install --upgrade --no-cache-dir "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --upgrade torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q xformers trl peft accelerate bitsandbytes

In [ ]:
# ── Cell 2: Install project dependencies ─────────────────────────────────────

!pip install -q pandas networkx tqdm matplotlib pyvis

In [ ]:
# ── Cell 3: Clone repository ──────────────────────────────────────────────────
# Replace USERNAME with your actual GitHub username before running.

!git clone https://github.com/USERNAME/auto-hkg.git
%cd auto-hkg

import sys
sys.path.insert(0, 'src')

print('Repository cloned. src/ added to Python path.')

In [ ]:
# ── Cell 4: Upload dataset ────────────────────────────────────────────────────
# Upload your semicolon-separated CSV file.
# Expected columns (exact names): Konteks ; Pertanyaan ; Level Kognitif
# The file will be moved to data/Knowledge_Base_Update.csv after upload.

import os
import shutil
from google.colab import files

os.makedirs('data', exist_ok=True)
uploaded = files.upload()
shutil.move(list(uploaded.keys())[0], 'data/Knowledge_Base_Update.csv')

print('Dataset moved to data/Knowledge_Base_Update.csv')

In [ ]:
# ── Cell 5: Inspect VRAM and select model ─────────────────────────────────────
# This cell reads the available GPU memory and prints a model recommendation.
# Free VRAM may vary depending on the Colab session state.
# Run this cell before loading the model to confirm sufficient headroom.

import torch

total_vram = torch.cuda.get_device_properties(0).total_memory / 1e9
free_vram  = torch.cuda.mem_get_info()[0] / 1e9
bf16_ok    = torch.cuda.is_bf16_supported()

print(f'GPU          : {torch.cuda.get_device_name(0)}')
print(f'VRAM total   : {total_vram:.1f} GB')
print(f'VRAM free    : {free_vram:.1f} GB')
print(f'bfloat16     : {"supported" if bf16_ok else "not supported"}')
print()

if free_vram >= 60:
    print('Recommendation: qwen3-32b-bf16  (full bf16, highest quality, A100-80GB)')
elif free_vram >= 25:
    print('Recommendation: qwen3-14b-bf16  (full bf16, ~28 GB, excellent quality)')
elif free_vram >= 18:
    print('Recommendation: qwen3-32b       (4-bit, ~20 GB, best 4-bit option on A100-40GB)')
elif free_vram >= 8:
    print('Recommendation: qwen3-14b       (4-bit, ~10 GB, recommended default)')
else:
    print('Recommendation: qwen3-8b        (4-bit, ~6 GB, fallback for limited VRAM)')

In [ ]:
# ── Cell 6: Run pipeline ──────────────────────────────────────────────────────
# Set MODEL_ALIAS to one of the aliases listed in Cell 5 or the notebook header.
# The pipeline will automatically detect GPU capabilities and configure dtype,
# quantization, and context length accordingly via llm_client.py.
#
# batch_size=10 is appropriate for A100. Increase to 20 if VRAM allows and
# inference speed is sufficient. Decrease to 5 for very large models.
#
# sleep_between_batches=0.0 because local inference has no API rate limits.

from auto_hkg import AutoHKG, HKGConfig

MODEL_ALIAS = 'qwen3-14b'   # Change this based on Cell 5 recommendation

cfg = HKGConfig(
    provider='unsloth',
    model=MODEL_ALIAS,
    data_path='data/Knowledge_Base_Update.csv',
    batch_size=10,
    sleep_between_batches=0.0,
)

pipeline = AutoHKG(cfg)
pipeline.run()

In [ ]:
# ── Cell 7: Monitor VRAM during pipeline ──────────────────────────────────────
# Run this cell in a separate tab while Cell 6 is executing.
# It prints allocated and reserved VRAM every 15 seconds for 20 iterations.
# Allocated VRAM = memory actively used by tensors.
# Reserved VRAM  = memory held by PyTorch allocator (includes fragmentation).

import torch
import time

for i in range(20):
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved()  / 1e9
    print(f'[{i+1:02d}] Allocated: {allocated:.2f} GB  |  Reserved: {reserved:.2f} GB')
    time.sleep(15)

In [ ]:
# ── Cell 8: View graph statistics ─────────────────────────────────────────────

import sys
sys.path.insert(0, 'src')
from visualize_graph import load_graph, print_stats

G = load_graph()
print_stats(G)

In [ ]:
# ── Cell 9: Static graph visualization ───────────────────────────────────────

from visualize_graph import plot_static
from IPython.display import Image

plot_static(G, max_nodes=80)
Image('output/graph/graph_static.png')

In [ ]:
# ── Cell 10: Interactive graph visualization ──────────────────────────────────
# Renders a zoomable, draggable graph in the notebook output.
# Increase max_nodes if the dataset is large enough to warrant a denser view.

from visualize_graph import plot_interactive
from IPython.display import HTML

plot_interactive(G, max_nodes=150)
HTML(open('output/graph/graph_interactive.html').read())

In [ ]:
# ── Cell 11: Download all output files ───────────────────────────────────────
# Packages the entire output/ directory into a zip file and triggers download.
# The archive includes:
#   output/graph/knowledge_graph.json  (full graph in node-link format)
#   output/graph/nodes.csv             (node table)
#   output/graph/edges.csv             (edge table)
#   output/graph/stats.json            (summary statistics)
#   output/logs/auto_hkg.log           (processing log)
#   output/logs/checkpoint.json        (checkpoint state)

import shutil
from google.colab import files

shutil.make_archive('auto_hkg_output', 'zip', 'output')
files.download('auto_hkg_output.zip')